In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# Force TensorFlow to use CPU — avoids cuDNN init issues on WSL2.
# Perch embedding extraction is fast enough on CPU; PyTorch head training uses GPU separately.
import tensorflow_hub as hub
import numpy as np

# model = hub.load('https://www.kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/4')
# total_params = sum(np.prod(v.shape) for v in model._variables)
# print(model.signatures)
# print(list(model.__dict__.keys()))
# print(total_params)

I0000 00:00:1777161429.656841   21110 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model = hub.load('https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2/2')
total_params = sum(np.prod(v.shape) for v in model._tf_var_leaves)
print(model.signatures)
print(list(model.__dict__.keys()))
print(total_params)

: 

## Step 1 – Load a 5-second .ogg chunk → numpy array at 32 kHz

Perch v2 expects `float32` waveform, shape `(batch, 160000)`, at **32 kHz**.

In [ ]:
import librosa
import numpy as np

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = '../data/train_audio/22930/iNat317238.ogg'  
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.45734373 0.5143107


## Step 2 – Extract embeddings with Perch v2

`model2.infer_tf` returns a dict with keys: `embedding` (1536-d), `label` (14795 logits), `spectrogram`, `spatial_embedding`.  
We only need `embedding`.

In [ ]:
import tensorflow as tf

infer = model.signatures['serving_default']

def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = tf.constant(waveform[np.newaxis, :])   # (1, 160000)
    out = infer(inputs=inp)
    return out['embedding'].numpy()[0]           # (1536,)

emb = extract_embedding(chunk)
print('Embedding shape:', emb.shape)  # (1536,)

## Step 3 – Pre-extract all embeddings (one-time, saves to .npy)

Running Perch on every batch during training is slow. Extract once, cache to disk, then train the head on raw numpy arrays — much faster.

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path

# Load your pre-built chunks dataframe (same one used for EfficientNet training)
chunks_df = pd.read_parquet('../data/chunks.parquet')   # columns: file_path, offset_sec, encoded_label

EMB_CACHE = Path('../data/perch_embeddings.npy')
LBL_CACHE = Path('../data/perch_labels.npy')

if not EMB_CACHE.exists():
    embeddings, labels = [], []
    for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df)):
        wav = load_chunk(row['file_path'], row['offset_sec'])
        emb = extract_embedding(wav)
        embeddings.append(emb)
        labels.append(row['encoded_label'])
    embeddings = np.stack(embeddings).astype(np.float32)
    labels     = np.array(labels, dtype=np.int64)
    np.save(EMB_CACHE, embeddings)
    np.save(LBL_CACHE, labels)
    print(f'Saved {len(embeddings)} embeddings → {EMB_CACHE}')
else:
    embeddings = np.load(EMB_CACHE)
    labels     = np.load(LBL_CACHE)
    print(f'Loaded {len(embeddings)} cached embeddings')

print('embeddings:', embeddings.shape)  # (N, 1536)
print('labels:    ', labels.shape)      # (N,)

## Step 4 – Train a lightweight PyTorch head

The head is just `Linear(1536 → 234)` with dropout. Perch weights stay frozen.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

NUM_CLASSES = 234
EMBED_DIM   = 1536
BATCH_SIZE  = 256
NUM_EPOCHS  = 20
LR          = 1e-3

# ---- Dataset from cached numpy arrays ----
X = torch.from_numpy(embeddings)   # (N, 1536) float32
y = torch.from_numpy(labels)       # (N,)      int64

dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
n_val   = int(0.1 * len(dataset))
n_test  = len(dataset) - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---- Lightweight head ----
class PerchHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(EMBED_DIM),
            nn.Dropout(0.3),
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
head = PerchHead().to(device)
print(head)
print(f'Head parameters: {sum(p.numel() for p in head.parameters()):,}')

In [ ]:
import mlflow, mlflow.pytorch

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

mlflow.set_experiment('birdclef2026-perch-head')

best_val_loss = float('inf')
with mlflow.start_run(run_name='perch_v2_head'):
    mlflow.log_params({'epochs': NUM_EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE,
                       'embed_dim': EMBED_DIM, 'num_classes': NUM_CLASSES})

    for epoch in range(NUM_EPOCHS):
        # --- train ---
        head.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- validate ---
        head.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = head(xb)
                val_loss += criterion(preds, yb).item()
                correct  += (preds.argmax(1) == yb).sum().item()
                total    += len(yb)

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        val_acc     = correct / total
        scheduler.step()

        mlflow.log_metrics({'train_loss': train_loss, 'val_loss': val_loss,
                            'val_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            mlflow.pytorch.log_model(head, 'best_model')

print('Training complete. Best val loss:', best_val_loss)